# Charge-conjugation symmetrization

```{autolink-concat}
```

If charge conjugation maps the final state of a three-body decay onto **itself**, the
strong interaction relates each decay chain to the chain of its charge-conjugate
resonance. In $J/\psi \to \eta\, p\, \bar p$, for instance, every $N^*$ appears twice:
once as $N^{*+} \to p\,\eta$ and once as $\bar N^{*-} \to \bar p\,\eta$. The two chains
are then **one** model component, with a single set of couplings and a fixed relative
sign $s$,

```{math}
:label: conjugate-coupling-relation
\mathcal{H}\left[\bar R, \dots\right] = s \, \mathcal{H}\left[R, \dots\right] \, ,
\qquad s = \pm 1 \, .
```

The sign is not a fit parameter: it follows from the quantum numbers of the decay alone.
The {mod}`~ampform_dpd.cparity` module derives it for an **arbitrary** three-body decay
and applies it to an {class}`~ampform_dpd.AmplitudeModel`.

## Where the sign comes from

Charge conjugation is unitary, the strong interaction conserves it, and the initial state
is an eigenstate, so the amplitude obeys $\mathcal{A}(f) = C_0 \, \mathcal{A}(Cf)$.
Pushing $C$ through each isobar vertex and multiplying along the chain, all the
unobservable conventional phases $c_a$ (defined by $C\left|a\right> = c_a
\left|\bar a\right>$) cancel: intermediate resonances telescope, particle–antiparticle
pairs in the final state cancel because $C^2 = 1$, and only the **measured** C-parities
of the self-conjugate final-state particles survive. What is left is

```{math}
:label: conjugate-coupling-sign
s
= \underbrace{C_0}_\text{initial state}
\; \underbrace{\prod_a C_a}_\text{self-conjugate finals}
\; \underbrace{\prod_v \eta_v}_\text{re-ordered vertices} \, .
```

The last product is pure bookkeeping. A two-particle state is a *constructed* object: one
particle is listed first and defines the direction of the relative momentum, and the
spins are coupled in the listed order. Charge conjugation hands us the image chain with
its particles in one order, while the code writes that chain in another, and each
mismatch costs a known phase. In the cyclic pair ordering $(ij)k \in
\left\{(23)1,(31)2,(12)3\right\}$ of the [DPD
paper](https://doi.org/10.1103/PhysRevD.101.034033) {cite}`JPAC:2019ufm`, the production
vertex $0 \to R\,k$ is always written as (resonance, spectator) and never mismatches; the
decay vertex $R \to i\,j$ mismatches whenever charge conjugation acts non-trivially on
$i$ or $j$. Its phase depends on the basis the couplings are defined in,

```{math}
:label: exchange-phase
\eta^\text{LS} = (-1)^{l+s_i+s_j-S} \, ,
\qquad
\eta^\text{helicity} = (-1)^{J_R-s_i-s_j} \, ,
```

with $(l, S)$ the $LS$ coupling of $R \to i\,j$ and $J_R$ the spin of the resonance.

:::{admonition} Never mix the two bases
:class: warning
$\eta^\text{LS}$ depends on $l$, $\eta^\text{helicity}$ does not, so the two bases
generally give **different** signs for the same resonance. Applying a sign derived in one
basis to the couplings of the other is the classic way to get this wrong.
:::

In [ ]:
from __future__ import annotations

import logging
import os
import warnings
from typing import NamedTuple

import attrs
import jax.numpy as jnp
import matplotlib.pyplot as plt
import qrules
import sympy as sp
from IPython.display import Latex, Markdown
from matplotlib.colors import CenteredNorm
from tensorwaves.data.transform import SympyDataTransformer

from ampform_dpd import DalitzPlotDecompositionBuilder
from ampform_dpd.adapter.qrules import normalize_state_ids, to_three_body_decay
from ampform_dpd.cparity import (
    get_c_forbidden_chains,
    get_conjugate_chain_pairs,
    get_conjugate_coupling_sign,
    get_conjugate_state_map,
    get_exchange_phase,
    is_c_symmetric,
    relate_conjugate_couplings,
    symmetrize_conjugate_couplings,
)
from ampform_dpd.dynamics.builder import formulate_breit_wigner_with_form_factor
from ampform_dpd.io import (
    as_markdown_table,
    aslatex,
    cached,
    mute_ampform_warnings,
    simplify_latex_rendering,
)

simplify_latex_rendering()
logging.getLogger("absl").setLevel(logging.ERROR)  # mute JAX
warnings.simplefilter("ignore", category=RuntimeWarning)

if STATIC_PAGE := "EXECUTE_NB" in os.environ:
    mute_ampform_warnings()

## Step 0: is there anything to derive?

Charge conjugation constrains a single amplitude only if it maps the final state onto
itself **as a set**. If it does, it permutes the final-state IDs, and that permutation is
what all the bookkeeping below follows from. Because $C$ is an involution, the permutation
of a three-body final state is either the identity (all final-state particles are
self-conjugate) or a single transposition.

In [ ]:
def generate_decay(final_state: list[str], resonances: list[str], **kwargs):
    reaction = qrules.generate_transitions(
        initial_state="J/psi(1S)",
        final_state=final_state,
        allowed_intermediate_particles=resonances,
        formalism="canonical-helicity",
        mass_conservation_factor=0,
        **kwargs,
    )
    return to_three_body_decay(normalize_state_ids(reaction.transitions), min_ls=True)


DECAY = generate_decay(
    ["eta", "p", "p~"],
    ["N(1535)", "N(1710)"],
    allowed_interaction_types="strong",
)
PIPIPI_DECAY = generate_decay(["pi0", "pi-", "pi+"], ["rho(770)", "f(2)(1270)"])
KAON_SIGMA_DECAY = generate_decay(
    ["K0", "Sigma+", "p~"],
    ["N(1700)+", "Sigma(1660)"],
    allowed_interaction_types="strong",
)

In [ ]:
def render_gate(decays: dict[str, object]) -> Markdown:
    rows = ["| decay | C-symmetric? | induced permutation |", "|:--|:-:|:--|"]
    for label, decay in decays.items():
        if is_c_symmetric(decay):
            state_map = get_conjugate_state_map(decay)
            names = {i: s.name for i, s in decay.final_state.items()}
            permutation = ", ".join(
                f"{i}: {names[i]} ↦ {names[j]}" for i, j in state_map.items()
            )
            rows.append(f"| {label} | ✅ | {permutation} |")
        else:
            rows.append(f"| {label} | ❌ | — (maps onto a *different* final state) |")
    return Markdown("\n".join(rows))


render_gate({
    R"$J/\psi \to \eta\, p\, \bar p$": DECAY,
    R"$J/\psi \to \pi^0 \pi^- \pi^+$": PIPIPI_DECAY,
    R"$J/\psi \to K^0 \Sigma^+ \bar p$": KAON_SIGMA_DECAY,
})

The last decay is the counter-example: charge conjugation turns $K^0 \Sigma^+ \bar p$ into
$\bar K^0 \bar\Sigma^- p$, a *different* process, so there is no relation between chains
of the same model and nothing to symmetrize.

## The sign, factor by factor

We take $J/\psi \to \eta\, p\, \bar p$ with two $N^*$ states of **opposite parity**, so
that the wave dependence of $\eta^\text{LS}$ becomes visible.

In [ ]:
Latex(aslatex(DECAY, with_jp=True))

In [ ]:
Markdown(as_markdown_table([DECAY.initial_state, *DECAY.final_state.values()]))

Every chain in subsystem&nbsp;2 is paired with the chain of the charge-conjugate
resonance in subsystem&nbsp;3, with identical $LS$ couplings on both nodes:

In [ ]:
def render_pairs(decay) -> Markdown:
    rows = [
        "| $R$ | $\\bar R$ | $J^P$ | subsystems | production $LS$ | decay $LS$ |",
        "|:--|:--|:-:|:-:|:-:|:-:|",
    ]
    for chain, conjugate in get_conjugate_chain_pairs(decay):
        parity = "+" if chain.resonance.parity > 0 else "-"
        rows.append(
            f"| ${chain.resonance.latex}$ | ${conjugate.resonance.latex}$"
            f" | ${sp.latex(chain.resonance.spin)}^{parity}$"
            f" | {chain.spectator.index}, {conjugate.spectator.index}"
            f" | $({chain.incoming_ls.L}, {sp.latex(chain.incoming_ls.S)})$"
            f" | $({chain.outgoing_ls.L}, {sp.latex(chain.outgoing_ls.S)})$ |"
        )
    return Markdown("\n".join(rows))


render_pairs(DECAY)

Now the three factors of Equation&nbsp;{eq}`conjugate-coupling-sign`. The $J/\psi$ supplies
$C_0 = -1$, the $\eta$ is the only final-state particle that charge conjugation leaves in
place and supplies $C_\eta = +1$, and the decay vertex $N^* \to p\,\eta$ is the single
re-ordered vertex:

In [ ]:
def render_sign_table(decay) -> Markdown:
    header = R"| $R$ | $C_0 \prod_a C_a$ | $\eta^\text{LS}$ | $\eta^\text{hel}$"
    header += R" | $s_\text{LS}$ | $s_\text{hel}$ | $P_R$ |"
    rows = [header, "|:--|:-:|:-:|:-:|:-:|:-:|:-:|"]
    for chain, _ in get_conjugate_chain_pairs(decay):
        s_ls = get_conjugate_coupling_sign(chain, "LS")
        s_hel = get_conjugate_coupling_sign(chain, "helicity")
        eta_ls = get_exchange_phase(chain, "LS")
        eta_hel = get_exchange_phase(chain, "helicity")
        rows.append(
            f"| ${chain.resonance.latex}$ | ${s_ls * eta_ls:+d}$"
            f" | ${eta_ls:+d}$ | ${eta_hel:+d}$"
            f" | $\\mathbf{{{s_ls:+d}}}$ | $\\mathbf{{{s_hel:+d}}}$"
            f" | ${chain.resonance.parity:+d}$ |"
        )
    return Markdown("\n".join(rows))


render_sign_table(DECAY)

Two things to read off this table.

1. In the $LS$ basis the whole expression collapses to the **parity of the resonance**,
   $s_\text{LS} = C_\psi C_\eta (-1)^l = P_{N^*}$, because the spin-0 $\eta$ fixes the
   decay spin to $S=\tfrac12$ and parity conservation fixes $(-1)^l = P_{N^*} P_p P_\eta =
   -P_{N^*}$. This is the result derived in
   [ComPWA/jpsi-nstar#573](https://github.com/ComPWA/jpsi-nstar/pull/573).
2. In the helicity basis the sign is $s_\text{hel} = -(-1)^{J-1/2}$, which is $-1$ for
   *both* $J=\tfrac12$ resonances. The two bases genuinely disagree for the
   $\tfrac12^+$ state.

In [ ]:
for chain, _ in get_conjugate_chain_pairs(DECAY):
    assert get_conjugate_coupling_sign(chain, "LS") == chain.resonance.parity

## Chains that are mapped onto themselves

If a chain sits in the subsystem that charge conjugation leaves in place, it is not tied
to another chain: Equation&nbsp;{eq}`conjugate-coupling-relation` relates its couplings to
**themselves**, which turns into a selection rule. The couplings have to vanish unless
$s=+1$.

$J/\psi \to \pi^0\pi^-\pi^+$ has both cases at once. The $\rho^\pm$ chains form a
conjugate pair, while the $\rho^0$ and $f_2(1270)$ chains recoil against the $\pi^0$ and
are mapped onto themselves.

In [ ]:
rows = [
    "| chain | subsystem | mapped onto | $s_\\text{LS}$ | verdict |",
    "|:--|:-:|:--|:-:|:--|",
]
state_map = get_conjugate_state_map(PIPIPI_DECAY)
paired = {
    c.resonance.name for pair in get_conjugate_chain_pairs(PIPIPI_DECAY) for c in pair
}
forbidden = {c.resonance.name for c in get_c_forbidden_chains(PIPIPI_DECAY)}
for chain in sorted(PIPIPI_DECAY.chains, key=lambda c: c.resonance.name):
    k = chain.spectator.index
    sign = get_conjugate_coupling_sign(chain, "LS")
    if state_map[k] != k:
        target, verdict = f"subsystem {state_map[k]}", "tied to its conjugate chain"
    elif chain.resonance.name in forbidden:
        target, verdict = "itself", "**forbidden**, coupling has to vanish"
    else:
        target, verdict = "itself", "allowed"
    rows.append(
        f"| ${chain.resonance.latex}$ | {k} | {target} | ${sign:+d}$ | {verdict} |"
    )
Markdown("\n".join(rows))

Both results have an independent cross-check.

- For the $\rho^\pm$ pair, $s = C_\psi C_{\pi^0} (-1)^{l=1} = +1$. **Isospin** gives the
  same answer: the $I=0$ combination of $\rho\pi$ has equal $\rho^+\pi^-$ and
  $\rho^-\pi^+$ coefficients.
- The selection rule says that an $X \to \pi^+\pi^-$ recoiling against the $\pi^0$ needs
  $C_X = C_\psi C_{\pi^0} = -1$, which allows the $\rho^0$ and forbids the $f_2(1270)$.
  **QRules** only produced the $f_2(1270)$ chain above because we let it use C-violating
  interactions; restricting it to the strong interaction removes exactly that chain.

In [ ]:
strong_decay = generate_decay(
    ["pi0", "pi-", "pi+"],
    ["rho(770)", "f(2)(1270)"],
    allowed_interaction_types="strong",
)
assert {c.resonance.name for c in strong_decay.chains} == {
    "rho(770)+",
    "rho(770)-",
    "rho(770)0",
}
assert get_c_forbidden_chains(strong_decay) == []
Markdown(
    "QRules keeps "
    + ", ".join(sorted(f"`{c.resonance.name}`" for c in strong_decay.chains))
    + " and drops `f(2)(1270)`, in agreement with the selection rule."
)

## Tying the couplings of a model

{func}`~ampform_dpd.cparity.relate_conjugate_couplings` turns the sign into substitutions
for an {class}`~ampform_dpd.AmplitudeModel`. By convention the sign is carried entirely by
the **production** coupling: only the product of the couplings along a chain is
observable, so how the sign is distributed over the two vertices is a choice, and putting
it on the production coupling keeps the decay couplings of a resonance and its charge
conjugate identical.

In [ ]:
model_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=False)
for chain in model_builder.decay.chains:
    model_builder.dynamics_choices.register_builder(
        chain, formulate_breit_wigner_with_form_factor
    )
model = model_builder.formulate(reference_subsystem=2)

In [ ]:
Latex(aslatex(relate_conjugate_couplings(model)))

{func}`~ampform_dpd.cparity.symmetrize_conjugate_couplings` applies them and drops the
now-dependent parameters, so that each conjugate pair shares one set of free couplings:

In [ ]:
tied_model = symmetrize_conjugate_couplings(model)
n_before = sum(1 for p in model.parameter_defaults if isinstance(p, sp.Indexed))
n_after = sum(1 for p in tied_model.parameter_defaults if isinstance(p, sp.Indexed))
Markdown(f"Couplings: **{n_before} → {n_after}**")

The same works for helicity couplings, for mixed bases, and for the single-coefficient
form of {meth}`.DalitzPlotDecompositionBuilder.formulate`. In the helicity basis the two
helicity indices of the decay node are swapped as well, because charge conjugation
delivers the decay products in the order in which the partner chain lists them the other
way round:

In [ ]:
helicity_builder = DalitzPlotDecompositionBuilder(DECAY, min_ls=True)
helicity_model = helicity_builder.formulate(reference_subsystem=2)
substitutions = relate_conjugate_couplings(helicity_model)
selection = {
    k: v
    for k, v in substitutions.items()
    if "decay" in str(k.base) and not k.indices[1].free_symbols
}
Latex(aslatex(dict(sorted(selection.items(), key=str)[:4])))

## What the sign does to the Dalitz plot

To see the sign at work we build models with a **single** conjugate pair and compare the
derived sign with the flipped one. The two subsystems are 2 and 3, so we plot $\sigma_3 =
M^2(\eta p)$ against $\sigma_2 = M^2(\bar p\eta)$: the $N^{*+}$ band is vertical, the
$\bar N^{*-}$ band is horizontal, and charge conjugation becomes a **mirror across the
diagonal**.

In [ ]:
SINGLE_DECAYS = {
    name: attrs.evolve(
        DECAY, chains=[c for c in DECAY.chains if name in c.resonance.name]
    )
    for name in ["N(1535)", "N(1710)"]
}


def create_model(decay, min_ls=False, dynamics=True):
    builder = DalitzPlotDecompositionBuilder(decay, min_ls=min_ls)
    if dynamics:
        for chain in builder.decay.chains:
            builder.dynamics_choices.register_builder(
                chain, formulate_breit_wigner_with_form_factor
            )
    return builder.formulate(reference_subsystem=2)


def tie_couplings(model, sign_flip=False):
    substitutions = relate_conjugate_couplings(model)
    if sign_flip:
        # Only the product of the couplings along a chain is observable, so the sign has
        # to be flipped on exactly one of the two vertices, not on both.
        substitutions = {
            symbol: -expr if "production" in str(symbol.base) else expr
            for symbol, expr in substitutions.items()
        }
    return attrs.evolve(
        model,
        amplitudes={k: v.xreplace(substitutions) for k, v in model.amplitudes.items()},
        parameter_defaults={
            k: v for k, v in model.parameter_defaults.items() if k not in substitutions
        },
    )

In [ ]:
class DalitzGrid(NamedTuple):
    """A regular grid over two of the Mandelstam variables of a decay."""

    i: int
    j: int
    X: jnp.ndarray
    Y: jnp.ndarray
    data: dict[str, jnp.ndarray]
    labels: dict[int, str]


def create_dalitz_grid(model, i, j, labels, resolution=400) -> DalitzGrid:
    k, *_ = {1, 2, 3} - {i, j}
    sigma_k, sigma_k_expr = list(model.invariants.items())[k - 1]
    m = sorted(model.masses, key=str)
    x_min = float(((m[j] + m[k]) ** 2).xreplace(model.masses))
    x_max = float(((m[0] - m[i]) ** 2).xreplace(model.masses))
    y_min = float(((m[i] + m[k]) ** 2).xreplace(model.masses))
    y_max = float(((m[0] - m[j]) ** 2).xreplace(model.masses))
    X, Y = jnp.meshgrid(
        jnp.linspace(x_min, x_max, num=resolution),
        jnp.linspace(y_min, y_max, num=resolution),
    )
    definitions = dict(model.variables)
    definitions[sigma_k] = sigma_k_expr
    definitions = {
        symbol: expr.xreplace(definitions).xreplace(model.masses)
        for symbol, expr in definitions.items()
    }
    data_transformer = SympyDataTransformer.from_sympy(definitions, backend="jax")
    data = {f"sigma{i}": X, f"sigma{j}": Y}
    data.update(data_transformer(data))
    return DalitzGrid(i, j, X, Y, data, labels)


GRID = create_dalitz_grid(
    model,
    i=3,
    j=2,
    labels={
        1: R"$\sigma_1 = M^2(p\bar p)$",
        2: R"$\sigma_2 = M^2(\bar p\eta)$",
        3: R"$\sigma_3 = M^2(\eta p)$",
    },
)

In [ ]:
def create_intensity_array(model, parameters=None, grid=None):
    grid = GRID if grid is None else grid
    expression = cached.unfold(model)
    substitutions = dict(model.parameter_defaults)
    substitutions.update(parameters or {})
    func = cached.lambdify(cached.xreplace(expression, substitutions), backend="jax")
    return func(grid.data)


derived_signs = {
    name: get_conjugate_coupling_sign(get_conjugate_chain_pairs(decay)[0][0], "LS")
    for name, decay in SINGLE_DECAYS.items()
}
intensities = {}
for name, decay in SINGLE_DECAYS.items():
    single_model = create_model(decay)
    intensities[name, "derived"] = create_intensity_array(tie_couplings(single_model))
    intensities[name, "flipped"] = create_intensity_array(
        tie_couplings(single_model, sign_flip=True)
    )

In [ ]:
%config InlineBackend.figure_formats = ['svg']

In [ ]:
def plot_dalitz(ax, array, title, grid=None, **kwargs):
    grid = GRID if grid is None else grid
    mesh = ax.pcolormesh(grid.X, grid.Y, array, rasterized=True, **kwargs)
    lo, hi = float(grid.X.min()), float(grid.X.max())
    ax.plot([lo, hi], [lo, hi], c="white", ls="dotted", lw=1)
    ax.set_aspect("equal")
    ax.set_title(title, fontsize=13)
    ax.set_xlabel(grid.labels[grid.i])
    ax.set_ylabel(grid.labels[grid.j])
    ax.figure.colorbar(mesh, ax=ax, pad=0.02, shrink=0.85)


resonance = "N(1710)"
derived = intensities[resonance, "derived"]
flipped = intensities[resonance, "flipped"]
scale = 1 / jnp.nansum(derived)
difference = (flipped - derived) * scale
halfrange = float(jnp.nanmax(jnp.abs(difference)))

plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(15, 4.4), ncols=3, layout="constrained")
sign = derived_signs[resonance]
plot_dalitz(axes[0], derived * scale, rf"$N(1710)$, derived sign ${sign:+d}$")
plot_dalitz(axes[1], flipped * scale, rf"$N(1710)$, flipped sign ${-sign:+d}$")
plot_dalitz(
    axes[2],
    difference,
    "difference",
    cmap="RdBu_r",
    norm=CenteredNorm(halfrange=halfrange),
)
plt.show()

:::{note}
Only the **product** of the couplings along a chain is observable, so "flipping the sign"
means flipping it on exactly one of the two vertices. Negating both substitutions would
flip the chain product twice and change nothing at all.
:::

The two bands are **identical** in both panels: the sign cannot touch the modulus of
either chain. What it changes is purely the *interference* between the two conjugate
chains, which is strongest where the two bands cross, on the diagonal. The difference is
itself mirror-symmetric, and so is each of the two intensities.

The interference does leak into the one-dimensional projections here, because this toy
model has nothing else in it:

In [ ]:
plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(13, 4), ncols=2, layout="constrained")
x = jnp.sqrt(GRID.X[0])
for ax, name in zip(axes, SINGLE_DECAYS, strict=True):
    sign = derived_signs[name]
    scale = 1 / jnp.nansum(intensities[name, "derived"])
    for key, style in [("derived", "-"), ("flipped", "--")]:
        label = f"$s={sign:+d}$ (derived)" if key == "derived" else f"$s={-sign:+d}$"
        ax.plot(
            x, jnp.nansum(intensities[name, key] * scale, axis=0), style, label=label
        )
    ax.set_title(f"${name.replace('N(', 'N(')}$", fontsize=13)
    ax.set_xlabel(R"$\sqrt{\sigma_3}$ (GeV)")
    ax.legend()
axes[0].set_ylabel("Normalized intensity (a.u.)")
plt.show()

## Mirror symmetry

Charge conjugation relabels the final state without touching a single momentum, so the
intensity of a C-conserving model has to be invariant under the induced permutation of
the Mandelstam variables. Leaving the conjugate couplings free breaks that invariance;
tying them restores it.

In [ ]:
helicity_model = create_model(DECAY, min_ls=True)
untied_parameters = {
    p: 1.8 + 0.9j
    for p in helicity_model.parameter_defaults
    if isinstance(p, sp.Indexed) and R"\overline{N}" in str(p)
}
intensity_untied = create_intensity_array(helicity_model, untied_parameters)
intensity_tied = create_intensity_array(symmetrize_conjugate_couplings(helicity_model))

In [ ]:
def mirror_asymmetry(array) -> float:
    return float(
        jnp.nanmax(jnp.abs(array - array.T)) / jnp.nanmax(jnp.abs(array + array.T))
    )


plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(11, 4.6), ncols=2, layout="constrained")
for ax, array, title in zip(
    axes,
    [intensity_untied, intensity_tied],
    ["untied couplings", "tied with the derived sign"],
    strict=True,
):
    plot_dalitz(
        ax,
        array / jnp.nansum(array),
        f"{title}\nmirror asymmetry: {mirror_asymmetry(array):.0e}",
    )
plt.show()

The residual asymmetry of the tied model is $\mathcal{O}(10^{-7})$, which is the single
precision that JAX uses by default — the tie is exact.

:::{admonition} A lone conjugate pair cannot arbitrate the sign
:class: important
Both sign choices give a mirror-symmetric intensity here — they are the two eigenstates of
the tie, and you can verify that by re-running the panel above with `sign_flip=True`. For a
model that contains **nothing but** a conjugate pair, mirror symmetry is a good check that
the couplings are tied to the *right partner*, with the right helicity indices, but it is
blind to $s$ itself.

That is a property of this toy model, not a general one. As soon as the model also
contains a chain that the mirror maps onto **itself**, that chain interferes with the pair
and the symmetry does become sensitive to $s$, as the next section shows.
:::

## Both mechanisms in one plot: $J/\psi \to 3\pi$

$J/\psi \to \pi^0\pi^-\pi^+$ puts a tied pair and a self-mapped chain on the *same* Dalitz
plot. With $1 = \pi^0$, $2 = \pi^-$ and $3 = \pi^+$, charge conjugation is again the
transposition $2 \leftrightarrow 3$, so plotting $\sigma_3 = M^2(\pi^0\pi^-)$ against
$\sigma_2 = M^2(\pi^+\pi^0)$ turns it into a mirror across the diagonal. The three $\rho$
bands then land in three different places:

- $\rho^-$ recoils against the $\pi^+$, so $\sigma_3 \approx m_\rho^2$ is a **vertical**
  band;
- $\rho^+$ recoils against the $\pi^-$, so $\sigma_2 \approx m_\rho^2$ is a **horizontal**
  band. The mirror exchanges the two: they are the conjugate pair, tied with $s=+1$;
- $\rho^0$ recoils against the $\pi^0$, and $\sigma_1 = \text{const}$ means
  $\sigma_2 + \sigma_3 = \text{const}$, an **anti-diagonal** band that the mirror maps onto
  itself. That is the chain the selection rule applies to.

The pions are light compared to the $J/\psi$, so the Dalitz region is almost the full
triangle and all three bands sit close to its edges.

In [ ]:
pipipi_model = create_model(strong_decay, min_ls=False)
pipipi_grid = create_dalitz_grid(
    pipipi_model,
    i=3,
    j=2,
    labels={
        1: R"$\sigma_1 = M^2(\pi^-\pi^+)$",
        2: R"$\sigma_2 = M^2(\pi^+\pi^0)$",
        3: R"$\sigma_3 = M^2(\pi^0\pi^-)$",
    },
)
pipipi_intensities = {
    key: create_intensity_array(
        tie_couplings(pipipi_model, sign_flip=flip), grid=pipipi_grid
    )
    for key, flip in [("derived", False), ("flipped", True)]
}

In [ ]:
derived = pipipi_intensities["derived"]
flipped = pipipi_intensities["flipped"]
scale = 1 / jnp.nansum(derived)
difference = (flipped - derived) * scale
rho_sign = get_conjugate_coupling_sign(
    get_conjugate_chain_pairs(strong_decay)[0][0], "LS"
)

plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(15, 4.4), ncols=3, layout="constrained")
for ax, array, sign, label in [
    (axes[0], derived, rho_sign, "derived"),
    (axes[1], flipped, -rho_sign, "flipped"),
]:
    plot_dalitz(
        ax,
        array * scale,
        f"$s={sign:+d}$ ({label})\nmirror asymmetry: {mirror_asymmetry(array):.0e}",
        grid=pipipi_grid,
    )
plot_dalitz(
    axes[2],
    difference,
    "difference",
    grid=pipipi_grid,
    cmap="RdBu_r",
    norm=CenteredNorm(halfrange=float(jnp.nanmax(jnp.abs(difference)))),
)
plt.show()

The derived sign keeps the intensity mirror-symmetric, but the flipped one does **not** —
it is asymmetric at the tens-of-percent level, while the derived one stays symmetric to
numerical precision, and the difference is largest where the $\rho^\pm$ bands cross the
$\rho^0$ band. That is exactly what the $\eta p \bar p$ toy model could not show: there
the model was *only* a conjugate pair, the mirror simply exchanged its two chains, and an
overall factor $s$ dropped out of the modulus. Here the $\rho^0$ chain survives the
reflection unchanged and acts as an interference **reference**, against which the relative
sign of the tied pair becomes observable.

Removing the $\rho^0$ again isolates the two ingredients, in both bases:

In [ ]:
charged_only = attrs.evolve(
    strong_decay,
    chains=[c for c in strong_decay.chains if c.resonance.name != "rho(770)0"],
)
rows = [
    R"| chains | basis | asymmetry, derived $s$ | asymmetry, flipped $s$ |",
    "|:--|:-:|--:|--:|",
]
for label, decay in [
    (R"$\rho^+\rho^-$ only", charged_only),
    (R"$\rho^+\rho^-\rho^0$", strong_decay),
]:
    for basis, min_ls in [("$LS$", False), ("helicity", True)]:
        sub_model = create_model(decay, min_ls=min_ls)
        sub_grid = create_dalitz_grid(
            sub_model, i=3, j=2, labels=pipipi_grid.labels, resolution=200
        )
        asymmetries = [
            mirror_asymmetry(
                create_intensity_array(
                    tie_couplings(sub_model, sign_flip=flip), grid=sub_grid
                )
            )
            for flip in (False, True)
        ]
        rows.append(
            f"| {label} | {basis} | {asymmetries[0]:.0e} | {asymmetries[1]:.0e} |"
        )
Markdown("\n".join(rows))

With the $\rho^0$ removed, **both** signs are symmetric to numerical precision and the
Dalitz plot really is blind to $s$. With it in, only the derived sign survives. The effect
is the same in the $LS$ and in the helicity basis, so it is not an artifact of the
coupling convention discussed next.

:::{warning}
This does not turn the Dalitz plot into a general sign-meter. It shows that a self-mapped
chain *can* make the intensity sensitive to $s$; whether it always does depends on the
fixed conventional phases of the following section, which this notebook does not
disentangle. The sign should still come from the derivation, with the Dalitz plot used as
a consistency check on top.
:::

## Convention caveats

Equation&nbsp;{eq}`conjugate-coupling-sign` relates the couplings as they are defined on
two-particle states in the pair ordering of the DPD paper. A real implementation stacks
more conventions on top of that — the $(-1)^{j_2-\lambda_2}$ particle-2 phases, the angle
conventions, the alignment rotations — and each of them can contribute a further fixed
phase that hand algebra does not see. Cross-check the tie against the amplitudes
themselves before using it in a fit.

One such convention bites right here, and it is worth spelling out.
{func}`~ampform_dpd.adapter.qrules.to_three_body_decay` sorts the children of an
{class}`.IsobarNode` by final-state ID, and
{meth}`.DalitzPlotDecompositionBuilder.formulate_subsystem_amplitude` builds the $LS$
Clebsch-Gordan factors from that order, while the isobar Wigner-$d$ function of the same
node uses the cyclic pair ordering $(31)2$. The two disagree for subsystem&nbsp;2, so that
subsystem picks up an extra $\eta^\text{LS}$ — see
[ComPWA/ampform-dpd#202](https://github.com/ComPWA/ampform-dpd/issues/202).

Since $\eta^\text{LS}$ depends on the wave, it cancels between resonances of the same $l$
but not between different ones, and the tie then gets the **relative** sign between the
two waves wrong. That is directly visible: the mirror symmetry the tie is supposed to
guarantee is broken in the $LS$ basis and intact in the helicity basis, which orders the
decay coupling cyclically and is therefore unaffected.

In [ ]:
ls_model = create_model(DECAY, min_ls=False)
intensity_ls = create_intensity_array(symmetrize_conjugate_couplings(ls_model))
intensity_hel = create_intensity_array(
    symmetrize_conjugate_couplings(create_model(DECAY, min_ls=True))
)

plt.rc("font", size=12)
fig, axes = plt.subplots(figsize=(11, 4.6), ncols=2, layout="constrained")
for ax, array, title in zip(
    axes,
    [intensity_ls, intensity_hel],
    [R"$LS$ basis, two waves", "helicity basis, two waves"],
    strict=True,
):
    asymmetry = (array - array.T) / jnp.nanmax(array + array.T)
    plot_dalitz(
        ax,
        asymmetry,
        f"{title}\nmirror asymmetry: {mirror_asymmetry(array):.0e}",
        cmap="RdBu_r",
        norm=CenteredNorm(halfrange=float(jnp.nanmax(jnp.abs(asymmetry)))),
    )
plt.show()

The helicity-basis model on the right is symmetric to machine precision, while the
$LS$-basis model on the left is not, even though the physics is the same. The asymmetry is
antisymmetric across the diagonal, as it must be.

Reordering the decay nodes cyclically before building the model removes it, which confirms
that this is a property of the amplitude builder rather than of the sign convention. It is
tracked by an `xfail`-marked test in `tests/test_cparity.py` and by
[#202](https://github.com/ComPWA/ampform-dpd/issues/202); until that is resolved, use
helicity couplings when you need the tie.